In [15]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow

RETRAIN = True
DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [30]:
#!docker pull ghcr.io/bencefarkas27/aggregator:test-auth
!docker run --name aggregator_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/bencefarkas27/aggregator:test-auth

b837e1114177201935bc8bd233181e0b092e0feb0a848e4aebe9d69200309f0d


### Healt endpoint check

In [33]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [34]:
run_ids = []

def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.InputLayer(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_client(client_id):
    # Load client data
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    y_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_labels.npy")
    # Create and train the model
    model = create_dummy_model(input_shape=x_train.shape[1:])
    model.fit(x_train, y_train, epochs=3)
    # Save the trained model weights
    model.save_weights(f"dummy_models/client_{client_id}.weights.h5")

def mlflow_log_client_model(client_id):
    weights_path = f"dummy_models/client_{client_id}.weights.h5"
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    input_shape = x_train.shape[1:]
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
        run_ids.append(run.info.run_id)


In [35]:
# Run these for client model training and logging to MLflow if no volume mounted client models exist for mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
for client_id in range(NUM_CLIENTS):
    if RETRAIN:
        train_client(client_id)
    mlflow_log_client_model(client_id)

Epoch 1/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9676 - loss: 2.0665
Epoch 2/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9872 - loss: 0.3510
Epoch 3/3
768/768 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9882 - loss: 0.2278


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_27', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/07 11:02:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/07 11:02:02 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/99905a1ddf4845e3b1ff1ca3df71db9c
🧪 View experiment at: http://localhost:5100/#/experiments/1
Epoch 1/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9460 - loss: 2.5568
Epoch 2/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9795 - loss: 0.4087
Epoch 3/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9856 - loss: 0.2380


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_30', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/07 11:02:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/07 11:02:13 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/5c448edcba57477b99a368e6fbed1427
🧪 View experiment at: http://localhost:5100/#/experiments/1
Epoch 1/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9720 - loss: 1.9889
Epoch 2/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9886 - loss: 0.5088
Epoch 3/3
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9919 - loss: 0.2521


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_33', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/07 11:02:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/07 11:02:25 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/18973a1257ea4d39bc9617c8690f82f1
🧪 View experiment at: http://localhost:5100/#/experiments/1


In [36]:
# Get run ids from experiment
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
runs_df = mlflow.search_runs([experiment.experiment_id])
run_ids = runs_df["run_id"].tolist()
run_ids.reverse()

runs = [f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)]
print(runs)
print(run_ids)

['runs:/99905a1ddf4845e3b1ff1ca3df71db9c/client_0_model', 'runs:/5c448edcba57477b99a368e6fbed1427/client_1_model', 'runs:/18973a1257ea4d39bc9617c8690f82f1/client_2_model']
['99905a1ddf4845e3b1ff1ca3df71db9c', '5c448edcba57477b99a368e6fbed1427', '18973a1257ea4d39bc9617c8690f82f1']


### Calling do_aggregate widt fedavg

In [37]:
aggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    "local_data_size": [
        len(np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy"))
        for client_id in range(NUM_CLIENTS)
    ],
    "aggregation_algorithm": "fedavg",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [38]:
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator:5000', 'aggregated_model_uri': 'models:/m-1fde8d4e989e4d2ba18abbf33c1b6e20', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


In [39]:
# Testing the client models vs the aggregated model
# Load test data
x_test = np.load(f"{DATA_PATH}/test_images.npy")
y_test = np.load(f"{DATA_PATH}/test_labels.npy")

In [40]:
# Pull the aggregated model from mlflow
try:
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    latest_model = models[0]
    aggregated_model = mlflow.tensorflow.load_model(f"models:/{latest_model.model_id}")
    aggregated_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print(f"Loaded latest aggregated model: {latest_model.model_id}")

    aggregated_loss, aggregated_accuracy = aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"Aggregated Model - Loss: {aggregated_loss}, Accuracy: {aggregated_accuracy}")
except Exception as e:
    print(f"Failed to load model: {e}")


Loaded latest aggregated model: m-1fde8d4e989e4d2ba18abbf33c1b6e20
Aggregated Model - Loss: 38.47157287597656, Accuracy: 0.2930000126361847


In [41]:
client_losses = []
client_accuracies = []
for client_id in range(NUM_CLIENTS):
    # Load the client model
    client_model = create_dummy_model(input_shape=x_test.shape[1:])
    client_model.load_weights(f"dummy_models/client_{client_id}.weights.h5")
    
    # Evaluate the client model
    client_loss, client_accuracy = client_model.evaluate(x_test, y_test, verbose=0)
    print(f"Client {client_id} Model - Loss: {client_loss}, Accuracy: {client_accuracy}")
    client_losses.append(client_loss)
    client_accuracies.append(client_accuracy)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_36', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 0 Model - Loss: 82.75552368164062, Accuracy: 0.414900004863739


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_37', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 1 Model - Loss: 152.79278564453125, Accuracy: 0.27799999713897705


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_38', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 2 Model - Loss: 229.202880859375, Accuracy: 0.2930000126361847


In [42]:
print(f"Aggregator's loss is {aggregated_loss} vs clients's losses: {client_losses} and avarage client loss: {np.mean(client_losses)}")
print(f"Aggregator's accuracy is {aggregated_accuracy} vs clients's accuracies: {client_accuracies} and avarage client accuracy: {np.mean(client_accuracies)}")

Aggregator's loss is 38.47157287597656 vs clients's losses: [82.75552368164062, 152.79278564453125, 229.202880859375] and avarage client loss: 154.91706339518228
Aggregator's accuracy is 0.2930000126361847 vs clients's accuracies: [0.414900004863739, 0.27799999713897705, 0.2930000126361847] and avarage client accuracy: 0.3286333382129669


### Calling do_aggregate widt deltarep

In [ ]:
aggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    
    "aggregation_algorithm": "deltarep",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [44]:
!docker stop aggregator_test

aggregator_test
